# Assignment 2 — Pandas: messy data handling

**Dataset source:** https://www.kaggle.com/datasets/danielesteves/rio-police-records
(file used: `BaseDPEvolucaoMensalCisp.csv` — monthly crime statistics per police district (CISP), State of Rio de Janeiro, from the Instituto de Segurança Pública)

**Before running:** upload `BaseDPEvolucaoMensalCisp.csv` using the file panel on the left of Colab.

In [1]:
import time, os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 2.2.3


## Q1. Load with explicit `dtype` and `parse_dates`; shrink memory

In [2]:
DATA_PATH = "BaseDPEvolucaoMensalCisp.csv"

dtypes = {
    "CISP": "int64", "AISP": "int64", "RISP": "int64", "mcirc": "int64",
    "munic": "object", "Regiao": "object",
    # these three come in with mixed types, so read them as text first
    "posse_drogas": "object", "trafico_drogas": "object", "apreensao_drogas_sem_autor": "object",
}

df = pd.read_csv(
    DATA_PATH,
    sep=";", encoding="latin-1",
    dtype=dtypes,
    parse_dates=["mes_ano"], date_format="%Ym%m",     # '2003m1' -> 2003-01-01
)
print("shape:", df.shape)
print("mes_ano dtype:", df["mes_ano"].dtype)

mem_before = df.memory_usage(deep=True).sum()
print(f"Memory BEFORE: {mem_before / 1024**2:.2f} MB")

shape: (26217, 63)
mes_ano dtype: object
Memory BEFORE: 19.92 MB


In [3]:
# downcast numeric columns
for c in df.select_dtypes(include="integer").columns:
    df[c] = pd.to_numeric(df[c], downcast="integer")
for c in df.select_dtypes(include="float").columns:
    df[c] = pd.to_numeric(df[c], downcast="float")

# low-cardinality text columns -> category
text_cols = df.select_dtypes(include=["object", "string"]).columns
low_card = [c for c in text_cols if df[c].nunique() / len(df) < 0.5]
for c in low_card:
    df[c] = df[c].astype("category")
print("converted to category:", low_card)

mem_after = df.memory_usage(deep=True).sum()
saving_pct = (1 - mem_after / mem_before) * 100
print(f"Memory AFTER : {mem_after / 1024**2:.2f} MB")
print(f"Saving       : {saving_pct:.1f}%")

converted to category: ['mes_ano', 'munic', 'Regiao', 'posse_drogas', 'trafico_drogas', 'apreensao_drogas_sem_autor']
Memory AFTER : 2.87 MB
Saving       : 85.6%


## Q2. Profiling DataFrame (one row per column)

In [4]:
def top_stats(s):
    vc = s.value_counts(dropna=True)
    if len(vc) == 0:
        return pd.Series({"most_frequent": np.nan, "frequency": 0})
    return pd.Series({"most_frequent": vc.index[0], "frequency": vc.iloc[0]})

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(),
}).join(df.apply(top_stats).T)

profile

,dtype,missing_count,missing_percent,unique_count,most_frequent,frequency
CISP,int16,0,0.0,138,1,196
mes,int8,0,0.0,12,4,2275
vano,int16,0,0.0,17,2016,1656
mes_ano,category,0,0.0,196,2016m2,138
AISP,int8,0,0.0,41,10,1769
...,...,...,...,...,...,...
indicador_letalidade,int8,0,0.0,46,0,8253
indicador_roubo_rua,int16,0,0.0,464,0,5770
indicador_roubo_veic,int16,0,0.0,253,0,7613
registro_ocorrencias,int16,0,0.0,1818,45,97


## Q3. Duplicates: exact rows and key columns

In [5]:
n_exact = df.duplicated().sum()
print("exact duplicate rows:", n_exact)

# One police district (CISP) should report once per month
keys = ["CISP", "mes_ano"]
n_key_dup = df.duplicated(subset=keys).sum()
print(f"duplicates on keys {keys}:", n_key_dup)

rows_before = len(df)
df = df.drop_duplicates(subset=keys, keep="first").reset_index(drop=True)
print("rows dropped:", rows_before - len(df), "| rows now:", len(df))

if n_key_dup == 0:
    print("Rule kept: key = CISP + mes_ano. It is already unique, so nothing was dropped.")
else:
    print("Rule kept: key = CISP + mes_ano, keep the first row, because a district can only have one record per month.")

exact duplicate rows: 0
duplicates on keys ['CISP', 'mes_ano']: 0
rows dropped: 0 | rows now: 26217
Rule kept: key = CISP + mes_ano. It is already unique, so nothing was dropped.


## Q4. Clean a text column (`Regiao`): strip, fix case, standardise labels

In [6]:
print("BEFORE")
print(df["Regiao"].astype("object").value_counts(dropna=False))

BEFORE
Regiao
Interior              8477
Capital               5192
Interior              4288
Baixada Fluminense    3616
Capital               2684
Grande Niterói        1320
Grande Niterói         640
Name: count, dtype: int64


In [7]:
region_raw = df["Regiao"].astype("object")
stripped = region_raw.str.strip()

# lower-case + remove accents to build a matching key
key = (stripped.str.casefold()
               .str.normalize("NFKD")
               .str.encode("ascii", "ignore")
               .str.decode("ascii"))

standard = {
    "capital": "Capital",
    "interior": "Interior",
    "baixada fluminense": "Baixada Fluminense",
    "grande niteroi": "Grande Niterói",
}
df["Regiao"] = key.map(standard).fillna(stripped.str.title()).astype("category")

print("AFTER")
print(df["Regiao"].astype("object").value_counts(dropna=False))

# same idea for the municipality column
m_before = df["munic"].nunique()
df["munic"] = df["munic"].astype("object").str.strip().astype("category")
print(f"\nunique municipality labels: {m_before} before -> {df['munic'].nunique()} after stripping spaces")

AFTER
Regiao
Interior              12765
Capital                7876
Baixada Fluminense     3616
Grande Niterói         1960
Name: count, dtype: int64

unique municipality labels: 253 before -> 90 after stripping spaces


## Q5. Regex: pull a number out of a text column

In [8]:
text_num_cols = ["posse_drogas", "trafico_drogas", "apreensao_drogas_sem_autor"]

for c in text_num_cols:
    extracted = df[c].astype("object").str.extract(r"(\d+)", expand=False)
    df[c + "_num"] = pd.to_numeric(extracted)
    print(f"{c:30s} rows with no number found: {df[c + '_num'].isna().sum()}")

df[[c for c in df.columns if c.endswith("_num")]].describe()

posse_drogas                   rows with no number found: 4646
trafico_drogas                 rows with no number found: 4646
apreensao_drogas_sem_autor     rows with no number found: 4646


,posse_drogas_num,trafico_drogas_num,apreensao_drogas_sem_autor_num
count,21571.000000,21571.000000,21571.000000
mean,4.498632,5.578230,1.382087
std,9.566338,7.524033,2.559001
min,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000
50%,2.000000,3.000000,1.000000
75%,5.000000,7.000000,2.000000
max,330.000000,83.000000,69.000000


## Q6. Parse the date column and build date features

In [9]:
parsed = pd.to_datetime(df["mes_ano"].astype(str).str.strip(), format="%Ym%m", errors="coerce")
print("dates that failed to parse:", parsed.isna().sum())
print("examples of labels that failed:", df.loc[parsed.isna(), "mes_ano"].astype(str).unique()[:10])

# rebuild the date from the year and month columns, which have no gaps
df["mes_ano"] = pd.to_datetime(dict(year=df["vano"], month=df["mes"], day=1))
print("missing dates after rebuilding:", df["mes_ano"].isna().sum())

df["year"] = df["mes_ano"].dt.year
df["month"] = df["mes_ano"].dt.month
df["day_of_week"] = df["mes_ano"].dt.day_name()
df["quarter"] = df["mes_ano"].dt.quarter

df[["mes_ano", "year", "month", "day_of_week", "quarter"]].head()

dates that failed to parse: 0
examples of labels that failed: []
missing dates after rebuilding: 0


,mes_ano,year,month,day_of_week,quarter
0,2003-01-01,2003,1,Wednesday,1
1,2003-01-01,2003,1,Wednesday,1
2,2003-01-01,2003,1,Wednesday,1
3,2003-01-01,2003,1,Wednesday,1
4,2003-01-01,2003,1,Wednesday,1


## Q7. `groupby` with named aggregation

In [10]:
def pct90(s):
    """90th percentile of a column."""
    return s.quantile(0.9)

summary = df.groupby("Regiao", observed=True).agg(
    n_records=("CISP", "count"),
    n_districts=("CISP", "nunique"),
    total_homicides=("hom_doloso", "sum"),
    mean_reports=("registro_ocorrencias", "mean"),
    p90_reports=("registro_ocorrencias", pct90),
).round(1)

summary

,n_records,n_districts,total_homicides,mean_reports,p90_reports
Regiao,,,,,
Baixada Fluminense,3616,19,28366,587.5,1205.5
Capital,7876,42,29183,709.0,1136.0
Grande Niterói,1960,10,9047,575.8,979.0
Interior,12765,67,20809,192.6,510.0


## Q8. Value minus group mean with `transform`

In [11]:
df["hom_vs_region_mean"] = (
    df["hom_doloso"] - df.groupby("Regiao", observed=True)["hom_doloso"].transform("mean")
)
df[["Regiao", "hom_doloso", "hom_vs_region_mean"]].head()

,Regiao,hom_doloso,hom_vs_region_mean
0,Capital,0,-3.705307
1,Capital,3,-0.705307
2,Capital,3,-0.705307
3,Capital,6,2.294693
4,Capital,4,0.294693


**Why `transform`:** it returns one value per original row, in the original order, so the group mean lines up with every row and can be subtracted directly.
`merge` would need an extra summary table and a join key, and could change the row order or row count.

## Q9. Pivot table, then melt back and prove it is lossless

In [12]:
pv = df.pivot_table(index="Regiao", columns="quarter", values="hom_doloso",
                    aggfunc="sum", observed=True)
print(pv)

long = pv.reset_index().melt(id_vars="Regiao", var_name="quarter", value_name="hom_doloso").dropna()

expected = (df.groupby(["Regiao", "quarter"], observed=True)["hom_doloso"]
              .sum().reset_index())

a = long.sort_values(["Regiao", "quarter"]).reset_index(drop=True).astype({"Regiao": "object"})
b = expected.sort_values(["Regiao", "quarter"]).reset_index(drop=True).astype({"Regiao": "object"})
pd.testing.assert_frame_equal(a, b, check_dtype=False)
print("\nround trip is lossless: melt(pivot) == groupby result")

quarter                1     2     3     4
Regiao                                    
Baixada Fluminense  7884  6990  6479  7013
Capital             8265  7220  6587  7111
Grande Niterói      2530  2231  2025  2261
Interior            5891  5121  4799  4998

round trip is lossless: melt(pivot) == groupby result


## Q10. Split into two frames, merge with inner / left / outer, and `validate`

In [13]:
facts = df[["CISP", "mes_ano", "hom_doloso", "registro_ocorrencias"]]
dims = df[["CISP", "munic", "Regiao"]].drop_duplicates("CISP")

# make the lookup table incomplete on purpose so the three joins differ:
# drop 10 districts that exist in `facts`, and add 2 districts that do not
extra = pd.DataFrame({"CISP": [9998, 9999], "munic": ["Test A", "Test B"], "Regiao": ["Capital", "Interior"]})
dims_part = pd.concat([dims.iloc[:-10].astype({"munic": "object", "Regiao": "object"}), extra], ignore_index=True)

inner = facts.merge(dims_part, on="CISP", how="inner")
left  = facts.merge(dims_part, on="CISP", how="left")
outer = facts.merge(dims_part, on="CISP", how="outer")

print("facts rows     :", len(facts))
print("dims_part rows :", len(dims_part))
print("inner rows     :", len(inner))
print("left rows      :", len(left))
print("outer rows     :", len(outer))
print("facts rows with no match in dims_part:", len(facts) - len(inner))
print("dims_part rows with no match in facts:", len(outer) - len(left))

facts rows     : 26217
dims_part rows : 130
inner rows     : 25078
left rows      : 26217
outer rows     : 26219
facts rows with no match in dims_part: 1139
dims_part rows with no match in facts: 2


**Why the counts differ:** `inner` keeps only rows whose `CISP` exists in both frames. `left` keeps every `facts` row, with empty `munic`/`Regiao` for the 10 dropped districts.
`outer` keeps everything from both sides, so it also adds the 2 extra districts (9998, 9999) that have no facts.

In [14]:
small = facts.head(200)      # CISP repeats in both frames, so this is a many-to-many join

try:
    facts.merge(small, on="CISP", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError raised:", e)

ok = facts.merge(small, on="CISP", validate="many_to_many")
print("validate='many_to_many' allows it, but rows explode:", len(facts), "x", len(small), "->", len(ok))

MergeError raised: Merge keys are not unique in right dataset; not a many-to-one merge
validate='many_to_many' allows it, but rows explode: 26217 x 200 -> 39200


## Q11. Per-group rolling mean and rank

In [15]:
df = df.sort_values(["CISP", "mes_ano"]).reset_index(drop=True)

df["reports_roll7"] = (
    df.groupby("CISP")["registro_ocorrencias"].transform(lambda s: s.rolling(7).mean())
)
df["reports_rank_in_cisp"] = (
    df.groupby("CISP")["registro_ocorrencias"].rank(ascending=False, method="min")
)

df.loc[df["CISP"] == df["CISP"].iloc[0],
       ["CISP", "mes_ano", "registro_ocorrencias", "reports_roll7", "reports_rank_in_cisp"]].head(10)

,CISP,mes_ano,registro_ocorrencias,reports_roll7,reports_rank_in_cisp
0,1,2003-01-01,578,NaN,144.0
1,1,2003-02-01,693,NaN,107.0
2,1,2003-03-01,573,NaN,145.0
3,1,2003-04-01,622,NaN,128.0
4,1,2003-05-01,669,NaN,115.0
5,1,2003-06-01,587,NaN,140.0
6,1,2003-07-01,641,623.285714,122.0
7,1,2003-08-01,637,631.714286,124.0
8,1,2003-09-01,656,626.428571,117.0
9,1,2003-10-01,703,645.000000,105.0


## Q12. The 3 most suspicious rows

In [16]:
cols = ["CISP", "mes_ano", "munic", "Regiao", "hom_doloso", "registro_ocorrencias"]
med = df["registro_ocorrencias"].median()

multi = df[df["munic"].astype(str).str.contains(";")]
i_multi = multi.index[0] if len(multi) else df["hom_doloso"].idxmax()
i_max = df["registro_ocorrencias"].idxmax()
zero = df[df["registro_ocorrencias"] == 0]
i_low = zero.index[0] if len(zero) else df["registro_ocorrencias"].idxmin()

suspicious = df.loc[[i_multi, i_max, i_low], cols]
display(suspicious)

print(f"Row {i_multi}: munic = '{df.at[i_multi, 'munic']}' - a cell should hold one municipality, not several joined by ';'.")
print(f"Row {i_max}: registro_ocorrencias = {df.at[i_max, 'registro_ocorrencias']}, the highest in the file (median is {med:.0f}) - a real spike or an entry error, it needs checking.")
print(f"Row {i_low}: registro_ocorrencias = {df.at[i_low, 'registro_ocorrencias']} for district {df.at[i_low, 'CISP']} - almost no reports in a whole month is unlikely.")

,CISP,mes_ano,munic,Regiao,hom_doloso,registro_ocorrencias
9052,55,2003-01-01,Queimados;Japeri,Baixada Fluminense,11,268
9968,59,2014-01-01,Duque de Caxias,Baixada Fluminense,7,2507
4181,26,2003-01-01,Rio de Janeiro,Capital,0,0


Row 9052: munic = 'Queimados;Japeri' - a cell should hold one municipality, not several joined by ';'.
Row 9968: registro_ocorrencias = 2507, the highest in the file (median is 359) - a real spike or an entry error, it needs checking.
Row 4181: registro_ocorrencias = 0 for district 26 - almost no reports in a whole month is unlikely.


## Q13. Write CSV and Parquet; compare size and load time

In [17]:
t0 = time.perf_counter(); df.to_csv("cleaned.csv", index=False);        t_csv_write = time.perf_counter() - t0
t0 = time.perf_counter(); df.to_parquet("cleaned.parquet", index=False); t_pq_write = time.perf_counter() - t0

size_csv = os.path.getsize("cleaned.csv") / 1024**2
size_pq  = os.path.getsize("cleaned.parquet") / 1024**2

t0 = time.perf_counter(); _ = pd.read_csv("cleaned.csv");         t_csv_read = time.perf_counter() - t0
t0 = time.perf_counter(); _ = pd.read_parquet("cleaned.parquet"); t_pq_read = time.perf_counter() - t0

result = pd.DataFrame({
    "file size (MB)": [size_csv, size_pq],
    "write time (s)": [t_csv_write, t_pq_write],
    "load time (s)":  [t_csv_read, t_pq_read],
}, index=["CSV", "Parquet"]).round(3)
print(result)
print(f"\nParquet is {size_csv / size_pq:.1f}x smaller and loads {t_csv_read / t_pq_read:.1f}x faster")

         file size (MB)  write time (s)  load time (s)
CSV               6.499           1.830          1.112
Parquet           1.046           0.633          0.246

Parquet is 6.2x smaller and loads 4.5x faster


## README.md (written automatically from this run)

In [18]:
readme = f'''# ML Assignment 2 — Pandas

Notebook: `assignment2_pandas.ipynb` (run top to bottom with Runtime -> Restart and run all).

## Dataset
Rio de Janeiro Crime Records (monthly statistics per police district, Instituto de Seguranca Publica)
Source: https://www.kaggle.com/datasets/danielesteves/rio-police-records
File used: `BaseDPEvolucaoMensalCisp.csv`

## Numbers
- Memory before: {mem_before / 1024**2:.2f} MB
- Memory after downcasting + category: {mem_after / 1024**2:.2f} MB
- **Memory saving: {saving_pct:.1f}%**
- CSV size: {size_csv:.2f} MB | Parquet size: {size_pq:.2f} MB
- CSV load time: {t_csv_read:.3f} s | Parquet load time: {t_pq_read:.3f} s
'''
with open("README.md", "w") as f:
    f.write(readme)
print(readme)

# ML Assignment 2 — Pandas

Notebook: `assignment2_pandas.ipynb` (run top to bottom with Runtime -> Restart and run all).

## Dataset
Rio de Janeiro Crime Records (monthly statistics per police district, Instituto de Seguranca Publica)
Source: https://www.kaggle.com/datasets/danielesteves/rio-police-records
File used: `BaseDPEvolucaoMensalCisp.csv`

## Numbers
- Memory before: 19.92 MB
- Memory after downcasting + category: 2.87 MB
- **Memory saving: 85.6%**
- CSV size: 6.50 MB | Parquet size: 1.05 MB
- CSV load time: 1.112 s | Parquet load time: 0.246 s

